<div dir="rtl">
<h1>🧪 Computer Vision — Intermediate Practical Lab</h1>
<h3>مختبر الرؤية الحاسوبية العملي — المستوى المتوسط</h3>
<hr>
<p style="direction: rtl; text-align: right;" ><strong>مبني مباشرة على:</strong> Week 1 (Python Review, NumPy &amp; Image Representation) و Week 2 (OpenCV &amp; Video Processing)</p>
<p style="direction: rtl; text-align: right;" ><strong>يفترض إتمام:</strong> <code>computer_vision_beginner_lab.ipynb</code> أولاً — هذا الدفتر يبني مباشرة على مفاهيمه.</p>
</div>


<div dir="rtl">
<h2>1. نظرة عامة على المختبر (Lab Overview)</h2>

<p style="direction: rtl; text-align: right;">هذا المختبر عبارة عن مجموعة تمارين عملية تفاعلية للمستوى <strong>المتوسط (Intermediate)</strong> فقط، مُشتقة مباشرة من تمارين الأسبوع الأول (Python Review, NumPy &amp; Image Representation) والأسبوع الثاني (OpenCV &amp; Video Processing).</p>

<p style="direction: rtl; text-align: right;">في المستوى المبتدئ تعاملت مع تمثيل الصور، تحميلها، وتغيير حجمها. في هذا المستوى ستبني على ذلك مباشرة عبر:</p>

<ul style="direction: rtl; text-align: right;">
<li>دمج عدة تحويلات هندسية في خط أنابيب واحد (Resize → Rotate → Crop).</li>
<li>تحويل RGB إلى Grayscale يدوياً بصيغة رياضية ومقارنته بدالة OpenCV الجاهزة.</li>
<li>ضبط السطوع على صورة حقيقية مع تفادي مشاكل الـ Overflow.</li>
<li>الرسم والتعليق على نسخة من الصورة دون إتلاف الأصلية.</li>
<li>حساب ورسم الهيستوغرام لمقارنة صورتين.</li>
<li>تطبيق عتبة ثنائية يدوية على مستند ممسوح ضوئياً.</li>
<li>تطبيع مصفوفة صورة تمهيداً لأي تحليل إحصائي.</li>
</ul>

<p style="direction: rtl; text-align: right;"><strong>تعليمات مهمة قبل البدء:</strong></p>

<ol style="direction: rtl; text-align: right;">
<li>حاول حل كل تمرين بنفسك في خلية "<strong>Your Solution</strong>" المخصصة <strong>قبل</strong> فتح الحل المرجعي.</li>
<li>الحلول موجودة لكنها <strong>مطوية (Collapsed)</strong> عن قصد — لا تفتحها إلا بعد محاولة حقيقية.</li>
<li>لا تكتفِ بنسخ الحل؛ الهدف هو أن تبني الفهم بنفسك.</li>
<li>استخدم خلية "<strong>Self-Check</strong>" للتأكد من صحة حلّك قبل الانتقال للتمرين التالي.</li>
</ol>

<p style="direction: rtl; text-align: right;"><strong>مسار التعلّم في هذا المختبر:</strong></p>

<pre style="direction: ltr; text-align: left; background-color: #f5f5f5; padding: 10px; border-radius: 5px;">
Geometric Pipeline (Resize + Rotate + Crop)
    ↓
Manual RGB→Grayscale vs cv2.cvtColor
    ↓
Brightness Adjustment (int16 + clip)
    ↓
Drawing & Annotation
    ↓
Histograms
    ↓
Manual Thresholding
    ↓
Normalization
</pre>

<hr>
</div>


<div dir="rtl">
<h2>2. أهداف التعلّم (Learning Objectives)</h2>

<p style="direction: rtl; text-align: right;">بنهاية هذا المختبر، ستكون قادراً على:</p>

<ul style="direction: rtl; text-align: right;">
<li>دمج <code>cv2.resize</code>, <code>cv2.getRotationMatrix2D</code>, و <code>cv2.warpAffine</code> في خط أنابيب واحد.</li>
<li>القص عبر Slicing بعد تطبيق تحويلات هندسية.</li>
<li>تطبيق صيغة تحويل RGB→Grayscale اليدوية <code>0.299R + 0.587G + 0.114B</code> بـ NumPy.</li>
<li>مقارنة نتيجة يدوية بنتيجة <code>cv2.cvtColor</code>.</li>
<li>استخدام <code>.astype(np.int16)</code> و <code>np.clip</code> لتفادي الـ Overflow عند ضبط السطوع.</li>
<li>الرسم فوق نسخة (<code >.copy()</code>) من صورة باستخدام <code>cv2.line</code>, <code>cv2.rectangle</code>, <code>cv2.putText</code>.</li>
<li>حساب هيستوغرام صورة باستخدام <code>cv2.calcHist</code> ورسمه.</li>
<li>تطبيق عتبة ثنائية ثابتة باستخدام <code>cv2.threshold(..., cv2.THRESH_BINARY)</code>.</li>
<li>تطبيع مصفوفة صورة إلى المجال [0, 1] واستخدام Boolean Masking لتحليلها.</li>
</ul>

<hr>
</div>


<div dir="rtl">
<h2>3. إعداد البيئة (Environment Setup)</h2>

<p style="direction: rtl; text-align: right;">سنستخدم نفس مكتبات المستوى المبتدئ: <code>numpy</code>, <code>cv2</code> (OpenCV), و <code>matplotlib</code>.</p>

<p style="direction: rtl; text-align: right;">نفّذ الخلية التالية لاستيراد المكتبات والتحقق من الإصدارات المثبّتة.</p>

<hr>
</div>


In [1]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)


NumPy: 2.5.1
OpenCV: 5.0.0


<div dir="rtl">
<blockquote style="direction: rtl; text-align: right; unicode-bidi: bidi-override; border-right: 4px solid #ccc; padding-right: 10px; margin-right: 0;">
<p style="direction: rtl; text-align: right; unicode-bidi: bidi-override;">إذا ظهرت الإصدارات بدون أي خطأ، فبيئتك جاهزة للانتقال إلى التمارين. 👍</p>
</blockquote>

<hr>
</div>


<div dir="rtl">
<h2>4. كيفية استخدام هذا الدفتر (How to Use This Notebook)</h2>

<p style="direction: rtl; text-align: right;">كل تمرين في هذا المختبر يتبع نفس الهيكل بالضبط:</p>

<pre style="direction: ltr; text-align: left; background-color: #f5f5f5; padding: 10px; border-radius: 5px; direction: ltr;">
Exercise Title
Topic / Difficulty / Objective / Scenario
Task / Requirements / Expected Result / Constraints
Hints 
────────────────────────
🧑‍💻 Your Solution   
────────────────────────
✅ Self-Check         
────────────────────────
▶ 💡 Show Solution    
</pre>

<p style="direction: rtl; text-align: right;"><strong>قاعدة ذهبية:</strong> لا تنتقل لفتح الحل قبل أن تحاول تشغيل حلّك الخاص أولاً ومحاولة اجتياز خلية Self-Check.</p>

<p style="direction: rtl; text-align: right;">تُرقَّم التمارين هنا من <strong>6</strong> إلى <strong>12</strong> استمراراً لترقيم التمارين 1-5 في دفتر المستوى المبتدئ.</p>

<hr>
</div>


<div dir="rtl">
<h2>📁 Image Resources — الصور المطلوبة لتمارين هذا المستوى</h2>

<p style="direction: rtl; text-align: right;">تمارين هذا المستوى تحتاج عدة صور حقيقية موجودة في نفس مجلد عمل هذا الدفتر (working directory):</p>

<ul style="direction: rtl; text-align: right;">
<li><strong><code>test.jpg</code></strong> — تُستخدم في التمارين 6، 7، 8، 9، و12 (يمكنك استخدام نفس صورة المستوى المبتدئ).</li>
<li><strong><code>day_time.jpg</code></strong> و <strong><code>night_time.jpg</code></strong> — صورتان بإضاءة مختلفة (نهار/ليل) للتمرين 10.</li>
<li><strong><code>scanned_document.jpg</code></strong> — صورة لمستند ممسوح ضوئياً للتمرين 11.</li>
</ul>


<p style="direction: rtl; text-align: right;">نفّذ الخلية التالية للتحقق من وجود الصور قبل المتابعة:</p>

<hr>
</div>


In [3]:
import os
import urllib.request

REQUIRED_IMAGES = ["test.jpg", "day_time.jpg", "night_time.jpg", "scanned_document.jpg"]
RAW_BASE_URL = "https://raw.githubusercontent.com/syriascitech/Medad-CV-Bootcamp/main/exercises/Week1"

for path in REQUIRED_IMAGES:
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(f"{RAW_BASE_URL}/{path}", path)
        except Exception as e:
            print(f"⚠️  Could not download '{path}': {e}")
    if os.path.exists(path):
        print(f"✅ Found '{path}'.")
    else:
        print(f"⚠️  Could not find '{path}' in the current directory.")
        print("   Please place the required image next to this notebook before running the related exercise.")


✅ Found 'test.jpg'.
✅ Found 'day_time.jpg'.
✅ Found 'night_time.jpg'.
✅ Found 'scanned_document.jpg'.


---



<div dir="rtl">
<h2>5. Exercise 6 — Geometric Transformation Pipeline (Resize → Rotate → Crop)</h2>
<h3>تمرين 6 — خط أنابيب تحويلات هندسية (Resize → Rotate → Crop)</h3>

<h3 style="direction: rtl;">Topic</h3>
<p style="direction: rtl; text-align: right;">Image Transformations (متعددة الخطوات).</p>

<h3 style="direction: rtl;">Difficulty</h3>
<p style="direction: rtl; text-align: right;">🟡 Intermediate</p>

<h3 style="direction: rtl;">Learning Objective</h3>
<p style="direction: rtl; text-align: right;">دمج عدة عمليات هندسية متتابعة في خط أنابيب واحد، تماماً كما يُطلب عند تجهيز صورة لنموذج ذكاء اصطناعي.</p>

<h3 style="direction: rtl;">Scenario</h3>
<p style="direction: rtl; text-align: right;">لديك صورة بأبعاد كبيرة جداً وتريد تجهيزها لإدخالها في نموذج ذكاء اصطناعي.</p>

<h3 style="direction: rtl;">Task</h3>
<p style="direction: rtl; text-align: right;">اكتب كوداً يقوم بالتالي:</p>
<ol style="direction: rtl; text-align: right;">
<li>غيّر أبعاد الصورة (Resize) لتصبح <strong>300×300</strong> بكسل، واحفظها في متغيّر باسم <code>resized</code>.</li>
<li>دوّر الصورة بزاوية 90 درجة باتجاه عقارب الساعة، واحفظها في متغيّر باسم <code>rotated</code>.</li>
<li>قصّ (Crop) منتصف الصورة تماماً ليصبح بأبعاد <strong>100×100</strong>، واحفظها في متغيّر باسم <code>cropped</code>.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li>استخدم <code>cv2.resize</code>, <code>cv2.getRotationMatrix2D</code>, و <code>cv2.warpAffine</code>.</li>
<li>استخدم الفهرسة (Slicing) للقص، وليس أي دالة جاهزة.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li><code>resized.shape == (300, 300, 3)</code>.</li>
<li><code>rotated.shape == (300, 300, 3)</code>.</li>
<li><code>cropped.shape == (100, 100, 3)</code>، تُظهر مركز الصورة بعد الدوران.</li>
</ul>

<h3 style="direction: rtl;">Constraints</h3>
<ul style="direction: rtl; text-align: right;">
<li>الزاوية الموجبة في <code>cv2.getRotationMatrix2D</code> تعني عكس عقارب الساعة، لذا للدوران باتجاه عقارب الساعة استخدم زاوية سالبة (-90).</li>
<li>لحساب مركز الصورة للقص استخدم <code>shape[1]//2</code> و <code>shape[0]//2</code>.</li>
</ul>

<hr>

<h3 style="direction: rtl;">🧑‍💻 Your Solution</h3>

<hr>
</div>



### Hints

<details>
<summary>💡 Hint 1</summary>

<code>M = cv2.getRotationMatrix2D(center, -90, 1.0)</code> ثم <code>cv2.warpAffine(img, M, (w, h))</code>.

</details>

<details>
<summary>💡 Hint 2</summary>

بعد الدوران بأبعاد 300×300، القص للمنتصف بأبعاد 100×100 يعني أخذ الشريحة <code>[100:200, 100:200]</code>.

</details>

---

### 🧑‍💻 Your Solution


In [ ]:
# TODO:
# 1. Resize the image to 300x300 -> store in `resized`.
# 2. Rotate `resized` 90 degrees clockwise -> store in `rotated`.
# 3. Crop the center 100x100 region of `rotated` -> store in `cropped`.

import cv2

img = cv2.imread('test.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Write your solution here




### ✅ Self-Check


In [ ]:
# Run this cell to validate your solution.

assert 'resized' in dir(), "Variable 'resized' not found. Did you name it correctly?"
assert resized.shape == (300, 300, 3), f"Expected resized shape (300, 300, 3), got {resized.shape}"
assert 'rotated' in dir(), "Variable 'rotated' not found. Did you name it correctly?"
assert rotated.shape == (300, 300, 3), f"Expected rotated shape (300, 300, 3), got {rotated.shape}"
assert 'cropped' in dir(), "Variable 'cropped' not found. Did you name it correctly?"
assert cropped.shape == (100, 100, 3), f"Expected cropped shape (100, 100, 3), got {cropped.shape}"

print("✅ Basic checks passed.")


### Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach
تطبيق العمليات الثلاث بالترتيب المطلوب تماماً: Resize، ثم Rotate، ثم Crop عبر Slicing.

#### Reference Implementation
```python
import cv2

img = cv2.imread('test.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# 1. Resize to 300x300
resized = cv2.resize(img_rgb, (300, 300))

# 2. Rotate 90 degrees clockwise (negative angle)
(h, w) = resized.shape[:2]
center = (w // 2, h // 2)
M = cv2.getRotationMatrix2D(center, -90, 1.0)
rotated = cv2.warpAffine(resized, M, (w, h))

# 3. Crop the center 100x100
start = (300 - 100) // 2  # = 100
cropped = rotated[start:start+100, start:start+100]

print(f"Final shape: {cropped.shape}")
```

#### Explanation
الترتيب مهم — القص يتم بعد الدوران وليس قبله، لأن المطلوب هو قص "منتصف الصورة" بعد أن أصبحت مُدوَّرة ومُغيَّرة الحجم.

#### Expected Result
مصفوفة نهائية بشكل (100, 100, 3).

#### Common Mistakes
- استخدام زاوية +90 بدلاً من -90 فيصبح الدوران عكس عقارب الساعة.
- حساب نقطة بداية القص بشكل خاطئ فتُنتج صورة غير مُمركزة.

</details>

---



<div dir="rtl">
<h2>6. Exercise 7 — Manual RGB→Grayscale vs OpenCV</h2>
<h3>تمرين 7 — تحويل RGB→Grayscale يدوياً مقابل OpenCV</h3>

<h3 style="direction: rtl;">Topic</h3>
<p style="direction: rtl; text-align: right;">تحويل الألوان (مقارنة).</p>

<h3 style="direction: rtl;">Difficulty</h3>
<p style="direction: rtl; text-align: right;">🟡 Intermediate</p>

<h3 style="direction: rtl;">Learning Objective</h3>
<p style="direction: rtl; text-align: right;">التحقق من أن الصيغة اليدوية لتحويل RGB إلى Grayscale (من الأسبوع الأول) تُعطي نفس نتيجة <code>cv2.cvtColor</code> تقريباً (الأسبوع الثاني).</p>

<h3 style="direction: rtl;">Task</h3>
<p style="direction: rtl; text-align: right;">اكتب كوداً يقوم بالتالي:</p>
<ol style="direction: rtl; text-align: right;">
<li>اقرأ صورة <code>test.jpg</code> وحوّلها إلى RGB.</li>
<li>طبّق صيغة التحويل اليدوي: <code>Gray = 0.299*R + 0.587*G + 0.114*B</code> باستخدام NumPy فقط (بدون OpenCV) واحصل على <code>dtype=uint8</code>. احفظها باسم <code>manual_gray</code>.</li>
<li>احسب النسخة الرمادية بواسطة <code>cv2.cvtColor(..., cv2.COLOR_RGB2GRAY)</code>. احفظها باسم <code>cv2_gray</code>.</li>
<li>قارن بين الصورتين بطباعة الفرق المطلق الأقصى بين المصفوفتين، واحفظه في متغيّر باسم <code>max_diff</code>.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li>استخرج القنوات R, G, B بالفهرسة <code>img_rgb[:,:,0]</code> إلخ، لا تستخدم <code>cv2.split</code>.</li>
<li>حوّل النتيجة إلى <code>uint8</code> باستخدام <code>.astype(np.uint8)</code>.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>صورتان رماديتان متشابهتان بصرياً.</li>
<li><code>max_diff</code> صغير جداً (عادة 0 أو 1 بسبب التقريب).</li>
</ul>

<h3 style="direction: rtl;">Constraints</h3>
<ul style="direction: rtl; text-align: right;">
<li>حوّل القنوات إلى <code>float</code> قبل الضرب لتفادي الـ overflow.</li>
<li>قارن القيم بعد تحويلها إلى <code>int</code> لتفادي مشاكل الطرح في <code>uint8</code>.</li>
</ul>

<hr>

<h3 style="direction: rtl;">🧑‍💻 Your Solution</h3>

<hr>
</div>



### Hints

<details>
<summary>💡 Hint 1</summary>

فصل القنوات: <code>R = img_rgb[:,:,0].astype(np.float64)</code> وهكذا للبقية لتفادي overflow أثناء الضرب.

</details>

<details>
<summary>💡 Hint 2</summary>

استخدم <code>np.max(np.abs(manual_gray.astype(int) - cv2_gray.astype(int)))</code> للمقارنة.

</details>

---

### 🧑‍💻 Your Solution


In [ ]:
# TODO:
# 1. Read 'test.jpg' and convert to RGB.
# 2. Manually compute grayscale using 0.299R + 0.587G + 0.114B -> `manual_gray` (dtype uint8).
# 3. Compute grayscale using cv2.cvtColor -> `cv2_gray`.
# 4. Compute and print the max absolute difference -> `max_diff`.

import cv2
import numpy as np

img = cv2.imread('test.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# Write your solution here




### ✅ Self-Check


In [ ]:
# Run this cell to validate your solution.

assert 'manual_gray' in dir(), "Variable 'manual_gray' not found. Did you name it correctly?"
assert manual_gray.dtype == np.uint8, f"Expected manual_gray dtype uint8, got {manual_gray.dtype}"
assert 'cv2_gray' in dir(), "Variable 'cv2_gray' not found. Did you name it correctly?"
assert manual_gray.shape == cv2_gray.shape, "manual_gray and cv2_gray must have the same shape."
assert 'max_diff' in dir(), "Variable 'max_diff' not found. Did you compute the max difference?"
assert max_diff <= 2, f"Expected a very small difference between manual and OpenCV grayscale, got {max_diff}"

print("✅ Basic checks passed.")


### Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach
تطبيق الصيغة اليدوية بنفس أسلوب تحدي الأسبوع الأول، ثم مقارنتها بدالة OpenCV الجاهزة من الأسبوع الثاني.

#### Reference Implementation
```python
import cv2
import numpy as np

img = cv2.imread('test.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

R = img_rgb[:, :, 0].astype(np.float64)
G = img_rgb[:, :, 1].astype(np.float64)
B = img_rgb[:, :, 2].astype(np.float64)

manual_gray = (0.299 * R + 0.587 * G + 0.114 * B).astype(np.uint8)

cv2_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

max_diff = np.max(np.abs(manual_gray.astype(int) - cv2_gray.astype(int)))
print(f"Max difference between manual and OpenCV grayscale: {max_diff}")
```

#### Explanation
التحويل الداخلي في OpenCV يستخدم نفس المعاملات تقريباً (مع تقريب مختلف قليلاً)، لذلك الفرق الأقصى المتوقع صغير جداً (0 أو 1) وليس صفراً بالضرورة بسبب التقريب الداخلي في OpenCV.

#### Expected Result
طباعة رقم صغير جداً يثبت تطابق الطريقتين تقريباً.

#### Common Mistakes
- عدم تحويل القنوات إلى float قبل الضرب، مما يسبب overflow في uint8.
- مقارنة القيم بدون تحويلها إلى int قبل الطرح (يسبب أخطاء عند القيم القريبة من الصفر في uint8).

</details>

---



<div dir="rtl">
<h2>7. Exercise 8 — Brightness Adjustment with Clipping on a Real Image</h2>
<h3>تمرين 8 — ضبط السطوع مع Clipping على صورة حقيقية</h3>

<h3 style="direction: rtl;">Topic</h3>
<p style="direction: rtl; text-align: right;">Pixel Manipulation (Brightness Adjustment).</p>

<h3 style="direction: rtl;">Difficulty</h3>
<p style="direction: rtl; text-align: right;">🟡 Intermediate</p>

<h3 style="direction: rtl;">Learning Objective</h3>
<p style="direction: rtl; text-align: right;">تطبيق تقنية زيادة السطوع مع تفادي الـ overflow (من الأسبوع الأول) على صورة حقيقية مُحمَّلة بـ OpenCV بدلاً من مصفوفة تجريبية صغيرة.</p>

<h3 style="direction: rtl;">Task</h3>
<p style="direction: rtl; text-align: right;">اكتب كوداً يقوم بالتالي:</p>
<ol style="direction: rtl; text-align: right;">
<li>اقرأ صورة <code>test.jpg</code> وحوّلها إلى Grayscale، واحفظها باسم <code>gray</code>.</li>
<li>زد سطوع الصورة بإضافة 40 لكل بكسل، مع التحويل إلى <code>int16</code> أولاً لتفادي الـ overflow، ثم استخدم <code>np.clip</code> لإعادة القيم إلى النطاق [0, 255]، ثم أعد التحويل إلى <code>uint8</code>. احفظها باسم <code>brighter</code>.</li>
<li>اطبع القيمة القصوى والدنيا في الصورة الأصلية والصورة المعدَّلة.</li>
<li>اعرض الصورتين جنباً إلى جنب.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li>استخدم <code>astype(np.int16)</code> قبل الجمع لتفادي التفاف القيم (wrap-around) في <code>uint8</code>.</li>
<li>أعد التحويل النهائي إلى <code>uint8</code> قبل العرض.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>صورة أكثر سطوعاً بشكل واضح.</li>
<li>أعلى قيمة في <code>brighter</code> لا تتجاوز 255.</li>
</ul>

<h3 style="direction: rtl;">Constraints</h3>
<ul style="direction: rtl; text-align: right;">
<li>لا تُضِف 40 مباشرة على مصفوفة <code>uint8</code> بدون تحويل النوع أولاً.</li>
</ul>

<hr>

<h3 style="direction: rtl;">🧑‍💻 Your Solution</h3>

<hr>
</div>



### Hints

<details>
<summary>💡 Hint 1</summary>

الخطوات: <code>astype(int16) → +40 → np.clip(0,255) → astype(uint8)</code> بالضبط كما في مثال الأسبوع الأول.

</details>

---

### 🧑‍💻 Your Solution


In [ ]:
# TODO:
# 1. Read 'test.jpg' and convert to Grayscale -> `gray`.
# 2. Increase brightness by 40 using int16 + np.clip -> `brighter` (dtype uint8).
# 3. Print min/max of both `gray` and `brighter`.
# 4. Display both images side by side.

import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('test.jpg')

# Write your solution here




### ✅ Self-Check


In [ ]:
# Run this cell to validate your solution.

assert 'gray' in dir(), "Variable 'gray' not found. Did you name it correctly?"
assert 'brighter' in dir(), "Variable 'brighter' not found. Did you name it correctly?"
assert brighter.dtype == np.uint8, f"Expected brighter dtype uint8, got {brighter.dtype}"
assert brighter.shape == gray.shape, "brighter and gray must have the same shape."
assert brighter.max() <= 255, "Values in brighter must not exceed 255."
assert brighter.astype(int).mean() >= gray.astype(int).mean(), "brighter should be, on average, brighter than gray."

print("✅ Basic checks passed.")


### Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach
نفس تقنية الأسبوع الأول لضبط السطوع، لكن على صورة حقيقية بدل مصفوفة 2×2 تجريبية.

#### Reference Implementation
```python
import cv2
import numpy as np
import matplotlib.pyplot as plt

img = cv2.imread('test.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

brighter = gray.astype(np.int16) + 40
brighter = np.clip(brighter, 0, 255).astype(np.uint8)

print(f"Original range: [{gray.min()}, {gray.max()}]")
print(f"Brighter range: [{brighter.min()}, {brighter.max()}]")

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(gray, cmap='gray'); plt.title('Original'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(brighter, cmap='gray'); plt.title('Brighter (+40)'); plt.axis('off')
plt.show()
```

#### Explanation
التحويل إلى int16 يسمح للقيم بتجاوز 255 مؤقتاً أثناء الجمع دون أن "تلتف" حول الصفر كما يحدث في uint8؛ ثم np.clip يقصّها إلى الحد الأقصى 255 قبل إعادتها إلى uint8.

#### Expected Result
صورة أفتح وضوحاً، وأعلى قيمة في الصورة الجديدة تساوي 255 وليس رقماً ملتفّاً من حول الصفر.

#### Common Mistakes
- إضافة 40 مباشرة على مصفوفة uint8 بدون تحويل النوع أولاً، مما يسبب التفاف القيم (مثلاً 250+40 يصبح رقماً صغيراً بدل 255).
- نسيان إعادة التحويل النهائي إلى uint8 قبل العرض.

</details>

---



<div dir="rtl">
<h2>8. Exercise 9 — Drawing and Annotating a Copy of the Image</h2>
<h3>تمرين 9 — الرسم والتعليق على نسخة من الصورة</h3>

<h3 style="direction: rtl;">Topic</h3>
<p style="direction: rtl; text-align: right;">Drawing & Annotation.</p>

<h3 style="direction: rtl;">Difficulty</h3>
<p style="direction: rtl; text-align: right;">🟡 Intermediate</p>

<h3 style="direction: rtl;">Learning Objective</h3>
<p style="direction: rtl; text-align: right;">ممارسة رسم أشكال هندسية ونصوص فوق صورة دون إتلاف الصورة الأصلية.</p>

<h3 style="direction: rtl;">Scenario</h3>
<p style="direction: rtl; text-align: right;">تريد تحديد منطقة في صورة وتوضيحها بعنوان نصي (مثلاً لتوضيح نتيجة لعرض تقديمي).</p>

<h3 style="direction: rtl;">Task</h3>
<p style="direction: rtl; text-align: right;">اكتب كوداً يقوم بالتالي:</p>
<ol style="direction: rtl; text-align: right;">
<li>اقرأ صورة <code>test.jpg</code> وحوّلها إلى RGB، واحفظها باسم <code>img_rgb</code>.</li>
<li>خذ نسخة (<code>.copy()</code>) من الصورة قبل الرسم عليها، واحفظها باسم <code>draw_img</code>.</li>
<li>ارسم مستطيلاً أخضر اللون حول منطقة اخترتها من الصورة على <code>draw_img</code>.</li>
<li>اكتب فوق المستطيل نصاً "Region of Interest" باستخدام <code>cv2.putText</code>.</li>
<li>اعرض الصورة الأصلية بجانب الصورة المُعلَّق عليها للتأكد أن الأصلية لم تتأثر.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li>خذ نسخة أولاً — دوال الرسم في OpenCV تُعدّل الصورة In-place.</li>
<li>استخدم ألوان بترتيب قنوات الصورة الممرَّرة (RGB هنا).</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>صورتان: <code>img_rgb</code> الأصلية بدون أي رسم، و <code>draw_img</code> تحتوي على مستطيل ونص فوقها.</li>
</ul>

<h3 style="direction: rtl;">Constraints</h3>
<ul style="direction: rtl; text-align: right;">
<li>لا ترسم مباشرة على <code>img_rgb</code>؛ ارسم فقط على <code>draw_img</code>.</li>
</ul>

<hr>

<h3 style="direction: rtl;">🧑‍💻 Your Solution</h3>

<hr>
</div>



### Hints

<details>
<summary>💡 Hint 1</summary>

<code>cv2.rectangle(draw_img, pt1, pt2, color, thickness)</code>.

</details>

<details>
<summary>💡 Hint 2</summary>

<code>cv2.putText(draw_img, text, org, font, fontScale, color, thickness)</code>.

</details>

---

### 🧑‍💻 Your Solution


In [ ]:
# TODO:
# 1. Read 'test.jpg' and convert to RGB -> `img_rgb`.
# 2. Copy it into `draw_img` before drawing.
# 3. Draw a green rectangle around a region of your choice on `draw_img`.
# 4. Write "Region of Interest" text above the rectangle using cv2.putText.
# 5. Display `img_rgb` and `draw_img` side by side.

import cv2
import matplotlib.pyplot as plt

img = cv2.imread('test.jpg')

# Write your solution here




### ✅ Self-Check


In [ ]:
# Run this cell to validate your solution.

assert 'img_rgb' in dir(), "Variable 'img_rgb' not found. Did you name it correctly?"
assert 'draw_img' in dir(), "Variable 'draw_img' not found. Did you name it correctly?"
assert draw_img.shape == img_rgb.shape, "draw_img and img_rgb must have the same shape."
assert not (draw_img == img_rgb).all(), "draw_img should differ from img_rgb (did you draw on it?)."

print("✅ Basic checks passed.")


### Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach
أخذ نسخة من الصورة، ثم الرسم عليها بدالتي rectangle و putText.

#### Reference Implementation
```python
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('test.jpg')
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

draw_img = img_rgb.copy()

cv2.rectangle(draw_img, (50, 50), (250, 200), (0, 255, 0), 3)
cv2.putText(draw_img, "Region of Interest", (50, 40),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(img_rgb); plt.title('Original (untouched)'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(draw_img); plt.title('Annotated Copy'); plt.axis('off')
plt.show()
```

#### Explanation
استخدام .copy() يضمن أن img_rgb الأصلية تبقى نظيفة، لأن دوال الرسم في OpenCV تعدّل المصفوفة الممرَّرة مباشرة.

#### Expected Result
الصورة الأصلية بدون أي أثر رسم، والنسخة الثانية تُظهر مستطيلاً أخضر مع تسمية نصية فوقه.

#### Common Mistakes
- الرسم مباشرة على img_rgb دون نسخ، مما يُتلف الصورة الأصلية لبقية الكود.
- وضع نص cv2.putText خارج حدود الصورة فلا يظهر.

</details>

---



<div dir="rtl">
<h2>9. Exercise 10 — Computing and Plotting Histograms for Two Images</h2>
<h3>تمرين 10 — حساب ورسم الهيستوغرام لصورتين</h3>

<h3 style="direction: rtl;">Topic</h3>
<p style="direction: rtl; text-align: right;">Histogram.</p>

<h3 style="direction: rtl;">Difficulty</h3>
<p style="direction: rtl; text-align: right;">🟡 Intermediate</p>

<h3 style="direction: rtl;">Learning Objective</h3>
<p style="direction: rtl; text-align: right;">فهم كيف يختلف شكل الهيستوغرام باختلاف ظروف الإضاءة، عبر مقارنة صورتين (نهار وليل مثلاً).</p>

<h3 style="direction: rtl;">Task</h3>
<p style="direction: rtl; text-align: right;">اكتب كوداً يقوم بالتالي:</p>
<ol style="direction: rtl; text-align: right;">
<li>حمّل صورتين: <code>day_time.jpg</code> و <code>night_time.jpg</code>.</li>
<li>حوّل كل واحدة إلى Grayscale.</li>
<li>احسب الهيستوغرام لكل صورة باستخدام <code>cv2.calcHist</code>. احفظهما في متغيّرين باسم <code>hist_day</code> و <code>hist_night</code>.</li>
<li>اعرض كل صورة بجانب الهيستوغرام الخاص بها في شبكة 2×2.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li>استخدم <code>cv2.calcHist([gray], [0], None, [256], [0, 256])</code> لكل صورة.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>هيستوغرام صورة النهار يميل نحو القيم الأعلى (أكثر إضاءة).</li>
<li>هيستوغرام صورة الليل يتركز في القيم المنخفضة (أغمق).</li>
</ul>

<h3 style="direction: rtl;">Constraints</h3>
<ul style="direction: rtl; text-align: right;">
<li>استخدم دالة مساعدة واحدة تُطبَّق على كل صورة لتفادي تكرار الكود.</li>
</ul>

<hr>

<h3 style="direction: rtl;">🧑‍💻 Your Solution</h3>

<hr>
</div>



### Hints

<details>
<summary>💡 Hint 1</summary>

استخدم دالة مساعدة واحدة تُطبَّق على كل صورة لتفادي تكرار الكود (نفس فكرة <code>plot_hist_side_by_side</code> في الدفتر).

</details>

---

### 🧑‍💻 Your Solution


In [ ]:
# TODO:
# 1. Load 'day_time.jpg' and 'night_time.jpg'.
# 2. Convert each to Grayscale.
# 3. Compute cv2.calcHist for each -> `hist_day`, `hist_night`.
# 4. Display each image next to its histogram in a 2x2 grid.

import cv2
import matplotlib.pyplot as plt

# Write your solution here




### ✅ Self-Check


In [ ]:
# Run this cell to validate your solution.

assert 'hist_day' in dir(), "Variable 'hist_day' not found. Did you name it correctly?"
assert 'hist_night' in dir(), "Variable 'hist_night' not found. Did you name it correctly?"
assert hist_day.shape == (256, 1), f"Expected hist_day shape (256, 1), got {hist_day.shape}"
assert hist_night.shape == (256, 1), f"Expected hist_night shape (256, 1), got {hist_night.shape}"
assert not (hist_day == hist_night).all(), "hist_day and hist_night should differ (different lighting conditions)."

print("✅ Basic checks passed.")


### Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach
بناء دالة عامة تحسب الهيستوغرام وتعرضه بجانب الصورة، ثم استدعاؤها مرتين.

#### Reference Implementation
```python
import cv2
import matplotlib.pyplot as plt

def show_image_and_histogram(filename, title, ax_img, ax_hist):
    img = cv2.imread(filename)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])

    ax_img.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax_img.set_title(title)
    ax_img.axis('off')

    ax_hist.plot(hist, color='black')
    ax_hist.fill_between(range(256), hist.flatten(), color='gray', alpha=0.3)
    ax_hist.set_title(f'Histogram — {title}')
    return hist

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
hist_day = show_image_and_histogram('day_time.jpg', 'Day', axes[0, 0], axes[0, 1])
hist_night = show_image_and_histogram('night_time.jpg', 'Night', axes[1, 0], axes[1, 1])
plt.tight_layout()
plt.show()
```

#### Explanation
cv2.calcHist يُنتج توزيعاً لعدد البكسلات لكل قيمة إضاءة من 0 إلى 255؛ صور الليل تُظهر تركيزاً في القيم المنخفضة، بينما صور النهار تمتد نحو القيم المرتفعة.

#### Expected Result
فرق واضح في موقع الذروة بين الهيستوغرامين.

#### Common Mistakes
- نسيان .flatten() عند استخدام fill_between مع نتيجة calcHist (تكون بشكل (256,1)).
- عدم تحويل الصورة لرمادي قبل حساب الهيستوغرام.

</details>

---



<div dir="rtl">
<h2>10. Exercise 11 — Manual Binary Thresholding on a Scanned Document</h2>
<h3>تمرين 11 — عتبة ثنائية يدوية على مستند ممسوح</h3>

<h3 style="direction: rtl;">Topic</h3>
<p style="direction: rtl; text-align: right;">Thresholding (يدوي).</p>

<h3 style="direction: rtl;">Difficulty</h3>
<p style="direction: rtl; text-align: right;">🟡 Intermediate</p>

<h3 style="direction: rtl;">Learning Objective</h3>
<p style="direction: rtl; text-align: right;">استخدام عتبة ثابتة لتحويل مستند ممسوح ضوئياً إلى أبيض وأسود نقي، للتخلص من الظلال وخلفية الورقة.</p>

<h3 style="direction: rtl;">Scenario</h3>
<p style="direction: rtl; text-align: right;">لديك صورة لورقة ممسوحة ضوئياً (Scanned Document).</p>

<h3 style="direction: rtl;">Task</h3>
<p style="direction: rtl; text-align: right;">اكتب كوداً يقوم بالتالي:</p>
<ol style="direction: rtl; text-align: right;">
<li>حوّل صورة <code>scanned_document.jpg</code> إلى التدرج الرمادي، واحفظها باسم <code>gray</code>.</li>
<li>طبّق عليها عتبة ثنائية <code>cv2.threshold</code> بحيث: أي بكسل قيمته الإضاءتية أعلى من 130 يتحول إلى أبيض (255)، وما دونه يتحول إلى أسود (0). احفظ النتيجة باسم <code>binary</code>.</li>
<li>اعرض الصورة الرمادية بجانب النتيجة الثنائية.</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li>استخدم <code>cv2.THRESH_BINARY</code> تحديداً (وليس <code>THRESH_BINARY_INV</code>).</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li>صورة أبيض وأسود نقية يسهل فيها تمييز النص عن خلفية الورقة.</li>
</ul>

<h3 style="direction: rtl;">Constraints</h3>
<ul style="direction: rtl; text-align: right;">
<li>استخدم القيمة 130 بالضبط كعتبة.</li>
</ul>

<hr>

<h3 style="direction: rtl;">🧑‍💻 Your Solution</h3>

<hr>
</div>



### Hints

<details>
<summary>💡 Hint 1</summary>

<code>_, binary = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY)</code>.

</details>

---

### 🧑‍💻 Your Solution


In [ ]:
# TODO:
# 1. Convert 'scanned_document.jpg' to Grayscale -> `gray`.
# 2. Apply cv2.threshold with threshold=130 using cv2.THRESH_BINARY -> `binary`.
# 3. Display the grayscale image next to the binary result.

import cv2
import matplotlib.pyplot as plt

img = cv2.imread('scanned_document.jpg')

# Write your solution here




### ✅ Self-Check


In [ ]:
# Run this cell to validate your solution.

import numpy as np

assert 'gray' in dir(), "Variable 'gray' not found. Did you name it correctly?"
assert 'binary' in dir(), "Variable 'binary' not found. Did you name it correctly?"
assert binary.shape == gray.shape, "binary and gray must have the same shape."
unique_vals = set(np.unique(binary).tolist())
assert unique_vals.issubset({0, 255}), f"binary should only contain 0 and 255, got {unique_vals}"

print("✅ Basic checks passed.")


### Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach
تطبيق عتبة ثابتة مباشرة على الصورة الرمادية.

#### Reference Implementation
```python
import cv2
import matplotlib.pyplot as plt

img = cv2.imread('scanned_document.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

_, binary = cv2.threshold(gray, 130, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1); plt.imshow(gray, cmap='gray'); plt.title('Grayscale'); plt.axis('off')
plt.subplot(1, 2, 2); plt.imshow(binary, cmap='gray'); plt.title('Binary (threshold=130)'); plt.axis('off')
plt.show()
```

#### Explanation
كل بكسل أعلى من 130 يصبح 255 (أبيض)، وكل ما دونه يصبح 0 (أسود)، مما يفصل الحبر الغامق عن خلفية الورقة الفاتحة.

#### Expected Result
نص واضح بالأسود على خلفية بيضاء نقية.

#### Common Mistakes
- استخدام THRESH_BINARY_INV عن طريق الخطأ فتنعكس الألوان.
- اختيار عتبة غير مناسبة للإضاءة الفعلية للصورة فتضيع تفاصيل النص.

</details>

---



<div dir="rtl">
<h2>11. Exercise 12 — Normalizing an Array Before Analysis</h2>
<h3>تمرين 12 — تطبيع (Normalize) مصفوفة قبل التحليل</h3>

<h3 style="direction: rtl;">Topic</h3>
<p style="direction: rtl; text-align: right;">Normalization.</p>

<h3 style="direction: rtl;">Difficulty</h3>
<p style="direction: rtl; text-align: right;">🟡 Intermediate</p>

<h3 style="direction: rtl;">Learning Objective</h3>
<p style="direction: rtl; text-align: right;">ممارسة تطبيع قيم مصفوفة صورة إلى المجال [0, 1]، وهي خطوة تحضيرية شائعة قبل أي تحليل إحصائي أو تمرير للصورة لنموذج تعلّم آلي.</p>

<h3 style="direction: rtl;">Task</h3>
<p style="direction: rtl; text-align: right;">اكتب كوداً يقوم بالتالي:</p>
<ol style="direction: rtl; text-align: right;">
<li>اقرأ صورة <code>test.jpg</code> وحوّلها إلى Grayscale.</li>
<li>طبّع قيم الصورة لتصبح بين 0 و1 باستخدام الصيغة <code>(arr - min) / (max - min)</code>. احفظ النتيجة باسم <code>normalized</code>.</li>
<li>باستخدام Boolean Masking على <code>normalized</code>، احسب نسبة البكسلات التي قيمتها أعلى من 0.5، واحفظها باسم <code>bright_ratio</code> (نسبة مئوية).</li>
</ol>

<h3 style="direction: rtl;">Requirements</h3>
<ul style="direction: rtl; text-align: right;">
<li>حوّل الصورة إلى <code>float</code> قبل التطبيع لتفادي القسمة الصحيحة.</li>
</ul>

<h3 style="direction: rtl;">Expected Result</h3>
<ul style="direction: rtl; text-align: right;">
<li><code>normalized</code> قيمها بين 0.0 و1.0.</li>
<li><code>bright_ratio</code> نسبة مئوية توضح مقدار السطوع النسبي في الصورة.</li>
</ul>

<h3 style="direction: rtl;">Constraints</h3>
<ul style="direction: rtl; text-align: right;">
<li>تجنّب القسمة على صفر إن كانت الصورة موحّدة اللون تماماً (max == min)، لكن لا حاجة لمعالجة هذه الحالة الحدّية هنا إلا إذا رغبت.</li>
</ul>

<hr>

<h3 style="direction: rtl;">🧑‍💻 Your Solution</h3>

<hr>
</div>



### Hints

<details>
<summary>💡 Hint 1</summary>

حوّل الصورة إلى float قبل التطبيع لتفادي القسمة الصحيحة.

</details>

<details>
<summary>💡 Hint 2</summary>

استخدم <code>mask = normalized > 0.5</code> ثم <code>mask.sum() / mask.size</code>.

</details>

---

### 🧑‍💻 Your Solution


In [ ]:
# TODO:
# 1. Read 'test.jpg' and convert to Grayscale.
# 2. Normalize its values to [0, 1] using (arr - min) / (max - min) -> `normalized`.
# 3. Compute the percentage of pixels brighter than 0.5 -> `bright_ratio`.

import cv2
import numpy as np

img = cv2.imread('test.jpg')

# Write your solution here




### ✅ Self-Check


In [ ]:
# Run this cell to validate your solution.

assert 'normalized' in dir(), "Variable 'normalized' not found. Did you name it correctly?"
assert normalized.min() >= 0.0 and normalized.max() <= 1.0, "normalized values must be within [0, 1]."
assert 'bright_ratio' in dir(), "Variable 'bright_ratio' not found. Did you name it correctly?"
assert 0.0 <= bright_ratio <= 100.0, f"bright_ratio should be a percentage between 0 and 100, got {bright_ratio}"

print("✅ Basic checks passed.")


### Solution

<details>
<summary>💡 Show Solution</summary>

#### Approach
تطبيق صيغة التطبيع من تمارين الأسبوع الأول، ثم استخدام Boolean Masking لتحليل النتيجة.

#### Reference Implementation
```python
import cv2
import numpy as np

img = cv2.imread('test.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float64)

min_val, max_val = gray.min(), gray.max()
normalized = (gray - min_val) / (max_val - min_val)

mask = normalized > 0.5
bright_ratio = mask.sum() / mask.size * 100
print(f"Percentage of pixels brighter than relative midpoint: {bright_ratio:.2f}%")
```

#### Explanation
التطبيع يعيد توزيع قيم الصورة الأصلية (0-255) إلى نطاق موحّد [0,1]، مما يسهّل مقارنة صور مختلفة الإضاءة. الـ Boolean Mask يُنتج مصفوفة من True/False يمكن استخدامها مباشرة للعد أو التصفية.

#### Expected Result
رقم نسبة مئوية يعكس مدى سطوع الصورة نسبياً.

#### Common Mistakes
- نسيان تحويل الصورة إلى float قبل الطرح والقسمة، مما يسبب أخطاء تقريب مع uint8.
- الخلط بين mask.sum() (عدد True) و mask.size (عدد كل العناصر).

</details>

---



<div dir="rtl">
<h2>12. Intermediate Skills Checklist</h2>
<h3>قائمة التحقق من مهارات المستوى المتوسط</h3>

<p style="direction: rtl; text-align: right;">راجع القائمة التالية وتأكد أنك قادر فعلاً على كل بند قبل الانتقال للمستوى التالي:</p>

<ul style="direction: rtl; text-align: right;">
<li>☐ أستطيع دمج <code>resize</code>, <code>getRotationMatrix2D</code>, و <code>warpAffine</code> في خط أنابيب واحد.</li>
<li>☐ أستطيع قصّ صورة بعد تحويلها هندسياً باستخدام Slicing.</li>
<li>☐ أفهم صيغة تحويل RGB→Grayscale اليدوية وأستطيع تطبيقها بـ NumPy.</li>
<li>☐ أستطيع مقارنة نتيجة يدوية بنتيجة <code>cv2.cvtColor</code>.</li>
<li>☐ أفهم لماذا نحوّل إلى <code>int16</code> قبل زيادة السطوع، ولماذا نستخدم <code>np.clip</code>.</li>
<li>☐ أستطيع الرسم فوق نسخة من صورة دون التأثير على الأصلية.</li>
<li>☐ أستطيع حساب هيستوغرام صورة باستخدام <code>cv2.calcHist</code> وتفسيره.</li>
<li>☐ أستطيع تطبيق عتبة ثنائية ثابتة بـ <code>cv2.threshold</code>.</li>
<li>☐ أستطيع تطبيع مصفوفة صورة إلى [0, 1] واستخدام Boolean Masking لتحليلها.</li>
</ul>

</div>


<div dir="rtl">
<h2>13. Completion Summary</h2>
<h3>ملخص الإنجاز</h3>

<h3 style="direction: rtl;">ما الذي تدرّبت عليه (What You Practiced)</h3>

<ul style="direction: rtl; text-align: right;">
<li>خطوط أنابيب التحويلات الهندسية (Geometric Transformation Pipelines).</li>
<li>تحويل الألوان اليدوي مقابل الجاهز (Manual vs. Built-in Color Conversion).</li>
<li>ضبط السطوع مع تفادي الـ Overflow (Brightness Adjustment with Clipping).</li>
<li>الرسم والتعليق على الصور (Drawing &amp; Annotation).</li>
<li>الهيستوغرام (Histograms).</li>
<li>العتبة الثنائية اليدوية (Manual Thresholding).</li>
<li>التطبيع (Normalization) وBoolean Masking.</li>
</ul>

<h3 style="direction: rtl;">هل أنت جاهز للمستوى التالي؟ (Ready for the Next Level?)</h3>

<p style="direction: rtl; text-align: right;">المستوى التالي (Advanced) سيبني مباشرة على ما تعلّمته هنا، وسيتضمن مفاهيم مثل:</p>

<ul style="direction: rtl; text-align: right;">
<li>عتبة Otsu التلقائية والعتبة التكيّفية (Otsu &amp; Adaptive Thresholding).</li>
<li>العمليات على الصور الثنائية (Binary Image Operations).</li>
<li>مطابقة الهيستوغرام (Histogram Matching).</li>
<li>معالجة الفيديو الحي (Live Video Processing).</li>
<li>تصحيح كود معطوب (Debugging).</li>
</ul>

<p style="direction: rtl; text-align: right;">هذه المواضيع <strong>لن تُشرح أو تُطبَّق في هذا الدفتر</strong> — هي فقط لإعطائك فكرة عمّا ينتظرك في المرحلة القادمة.</p>

<hr>

<p style="direction: rtl; text-align: right;"><strong>🎉 مبروك على إتمام مختبر المستوى المتوسط!</strong></p>

</div>
